# Reading the Fingerprints of Yield Loss: Wafer-Map Defect Pattern Analysis on WM-811K
### 811,457 wafer bin maps from 46,293 production lots — which spatial patterns kill the most dies, and can we recognise them automatically?

**Why this matters.** After fabrication every die on a wafer is electrically probed and marked pass or fail. The resulting *wafer bin map* is a fingerprint: failing dies that cluster in the centre, along the edge, in a ring or along a line point to different process steps and tools. Recognising the pattern quickly tells an engineer where to look for the root cause, and quantifying which patterns cost the most dies tells them what to fix first.

**What this notebook does**
1. Audits the raw data (labels, map sizes, duplicates, conflicting labels).
2. Quantifies **yield loss per pattern** (which patterns are the "killer" defects) and tests whether patterns **cluster within lots** and **by slot position**, which separates systematic tool/lot problems from random ones.
3. Engineers **interpretable spatial features** (radial profile, 13-region density, Radon-transform projections, geometry of the largest failing cluster, spatial clustering of failures).
4. Trains Random Forest and Gradient Boosting classifiers on a **lot-aware split** (no lot in both training and test), compares them with a raw-pixel baseline, and shows how a random split would have overstated accuracy.
5. Analyses errors and feature importance, and applies the model to **unlabeled production wafers**.

Everything runs on a laptop CPU (no GPU, no deep-learning framework). Numbers in the text are computed live by the cells, so they always match the data you run it on.

## Contents
1. Problem statement and guiding questions
2. Data provenance and data dictionary
3. Reproducible setup and data loading
4. Data-quality audit
5. Cleaning decisions and cleaning log
6. What each pattern looks like
7. Q1 — Which patterns cost the most dies?
8. Q2 — Are defects systematic? Lot clustering and slot position
9. Feature engineering: turning a map into numbers an engineer can read
10. Q3 — Can a model recognise the patterns on unseen lots?
11. Error analysis and what the model relies on
12. Q4 — What does the model see in unlabeled production wafers?
13. Answers, recommendations and limitations
14. Reproducibility record

## 1. Problem statement and guiding questions

| # | Question | Why it matters in a fab |
|---|---|---|
| Q1 | Which defect patterns cause the most die loss in total? | Prioritise engineering effort on the patterns that cost the most |
| Q2 | Do patterns cluster within lots or at particular slot positions? | Lot clustering points to a systematic tool or process cause, not random bad luck |
| Q3 | Can a model recognise the pattern of a wafer from a lot it has never seen? | Automatic classification only helps if it works on new production |
| Q4 | What does the model find in the ~79% of wafers nobody labelled? | Estimates the real prevalence of each pattern in production |

**Pattern → commonly associated root causes.** These are the associations most often reported in the wafer-map literature and in fab practice. They are *starting points for an investigation*, not diagnoses: confirming a cause needs the process and tool history, which this dataset does not include.

| Pattern | What it looks like | Commonly associated causes |
|---|---|---|
| Center | Failing dies concentrated at the wafer centre | Centre-to-edge non-uniformity in deposition, etch or CMP; gas-flow or temperature profile issues |
| Donut | Ring of failures between centre and edge | Radial non-uniformity, e.g. thermal (bake/hot-plate) or deposition profile |
| Edge-Ring | Failures all around the edge | Edge effects in etch/deposition, edge-bead removal, edge temperature in RTP |
| Edge-Loc | Failures in one part of the edge | Handling or clamping damage, chuck contact, local edge chipping |
| Loc | A local cluster away from the edge | Particles or local contamination, a local equipment defect |
| Scratch | A thin line or arc of failures | Mechanical damage: wafer handling, CMP scratch |
| Random | Failures spread uniformly | Random particles, environment contamination |
| Near-full | Almost every die fails | Gross process excursion: wrong recipe, tool failure, mis-processing |

## 2. Data provenance and data dictionary

| Item | Detail |
|---|---|
| Dataset | **WM-811K** (Wu, Jang & Chen, "Wafer Map Failure Pattern Recognition and Similarity Ranking for Large-Scale Data Sets", *IEEE Trans. Semiconductor Manufacturing* 28(1), 2015). Real production wafers; labels assigned by domain experts. |
| File | `LSWMD.pkl` (about 2 GB) |
| Where to get it | Kaggle: `https://www.kaggle.com/datasets/qingyi/wm811k-wafer-map` (free account). **Automatic:** if `data/LSWMD.pkl` is missing, the loading cell downloads it with your Kaggle API key (from `KAGGLE_USERNAME`/`KAGGLE_KEY`, `~/.kaggle/kaggle.json`, or a hidden prompt; the key is never stored in the notebook). **Manual:** download, unzip, and put `LSWMD.pkl` in `data/`. Original host: MIR Lab, `http://mirlab.org/dataSet/public/` |
| Hardware | Loading the raw file needs about 4–6 GB of free RAM **once**. The notebook then writes a compact cache (`data/wm811k_cache.pkl`) and later runs load that in seconds. |

| Field | Meaning | Notes |
|---|---|---|
| `waferMap` | 2-D array per wafer: 0 = no die (off-wafer), 1 = passing die, 2 = failing die | Map size varies with die size (from about 6×21 up to 300×202) |
| `dieSize` | Number of dies on the wafer | |
| `lotName` | Production lot | A lot is normally 25 wafers processed together |
| `waferIndex` | Slot position of the wafer in its lot (1–25) | |
| `failureType` | Expert label: `none` or one of 8 patterns; empty for unlabeled wafers | Stored as a nested array, e.g. `[['Edge-Ring']]` |
| `trianTestLabel` | The original authors' `Training`/`Test` split (column name misspelled in the source) | Empty for unlabeled wafers |

## 3. Reproducible setup and data loading

**Restart the kernel and run all cells in order.** The configuration cell holds every threshold and sample size. With the default settings the full notebook runs in roughly 10–20 minutes on a typical laptop (most of it the one-time load of the 2 GB file and the feature extraction). To try it quickly first, set `CONFIG['quick'] = True` (smaller samples, about 3–5 minutes).

Required packages: `numpy`, `pandas`, `scipy`, `scikit-learn`, `scikit-image`, `matplotlib`, `seaborn`, `joblib` (`pip install scikit-image` if it is missing).

In [ ]:
import os, sys, gc, json, time, pickle, hashlib, platform, warnings
from pathlib import Path
from collections import Counter
import numpy as np
import pandas as pd
import scipy
from scipy import stats
from scipy.ndimage import median_filter, convolve
from scipy.interpolate import interp1d
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
import seaborn as sns
import sklearn
import skimage
from skimage.transform import radon, resize
from skimage.measure import label as cc_label, regionprops
from joblib import Parallel, delayed
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import f1_score, balanced_accuracy_score, accuracy_score, confusion_matrix, classification_report
from sklearn.inspection import permutation_importance
from IPython.display import display, Markdown

warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=RuntimeWarning)
pd.set_option('display.max_columns', 40); pd.set_option('display.width', 200)
pd.set_option('display.float_format', lambda v: f'{v:,.4g}')

CONFIG = {
    'raw_file': 'LSWMD.pkl',
    'kaggle_dataset': 'qingyi/wm811k-wafer-map',
    'cache_file': 'wm811k_cache.pkl',
    'seed': 811,
    'quick': False,                 # True = smaller samples for a fast first run
    'min_dies': 100,                # wafers with fewer dies are too coarse for spatial features
    'max_none': 25_000,             # 'none' wafers kept for modelling (there are ~147k; the rest add little)
    'n_unlabeled': 30_000,          # unlabeled wafers sampled for the production analysis (Q4)
    'test_lots': 0.20, 'val_lots': 0.15,   # shares of LOTS (not wafers) held out
    'radon_angles': 60, 'radon_points': 20,
    'pixel_size': 32,               # raw-pixel baseline resolution
    'fdr_alpha': 0.05,
    'confidence': 0.70,             # minimum predicted probability to count an unlabeled wafer in Q4
    'n_jobs': -1,
}
if CONFIG['quick']:
    CONFIG.update({'max_none': 6_000, 'n_unlabeled': 5_000})
RNG = np.random.default_rng(CONFIG['seed'])

HERE = Path.cwd()
DATA_DIR = HERE / 'data'
OUT = HERE / 'outputs'
FIG = OUT / 'figures'
for d in (DATA_DIR, OUT, FIG): d.mkdir(parents=True, exist_ok=True)

PATTERNS = ['Center', 'Donut', 'Edge-Loc', 'Edge-Ring', 'Loc', 'Near-full', 'Random', 'Scratch']
CLASSES = ['none'] + PATTERNS
PAL = dict(zip(CLASSES, ['#999999', '#E69F00', '#56B4E9', '#009E73', '#F0E442', '#0072B2', '#D55E00', '#CC79A7', '#000000']))
WAFER_CMAP = ListedColormap(['white', '#BFD7EA', '#C0392B'])     # off-wafer, pass, fail
sns.set_theme(style='whitegrid', context='notebook', font_scale=0.95)
plt.rcParams.update({'figure.dpi': 100, 'axes.titleweight': 'bold', 'axes.titlesize': 11})

def savefig(fig, name):
    fig.savefig(FIG / f'{name}.png', dpi=150, bbox_inches='tight')
    plt.show()

def note(text):
    display(Markdown(text))

def bh(p):
    p = np.asarray(p, float); n = len(p); order = np.argsort(p)
    ranked = p[order] * n / np.arange(1, n + 1)
    q = np.minimum.accumulate(ranked[::-1])[::-1]
    out = np.empty(n); out[order] = np.minimum(q, 1.0)
    return out

def show_wafer(ax, m, title=None):
    ax.imshow(m, cmap=WAFER_CMAP, vmin=0, vmax=2, interpolation='nearest')
    ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)
    if title: ax.set_title(title, fontsize=8)

print(f'Python {platform.python_version()} | numpy {np.__version__} | pandas {pd.__version__} | scipy {scipy.__version__} | '
      f'scikit-learn {sklearn.__version__} | scikit-image {skimage.__version__} | matplotlib {matplotlib.__version__}')
print('Working folder:', HERE)

**Loading.** `LSWMD.pkl` was written in 2015 with an old version of pandas, so recent pandas versions sometimes cannot open it directly. The loader tries three strategies in turn: plain `pd.read_pickle`; the same after aliasing the old module names; and a custom unpickler that maps old pandas class paths to their current locations. Labels are unpacked from their nested-array form into plain strings, and a compact cache is written so the slow step runs only once.

In [ ]:
def unpack(v):
    """failureType / trianTestLabel are nested arrays such as [['none']] or empty arrays: return a plain string."""
    a = np.asarray(v, dtype=object).ravel()
    return str(a[0]).strip() if a.size else ''

class _CompatUnpickler(pickle.Unpickler):
    RENAMES = {'pandas.indexes.base': 'pandas.core.indexes.base', 'pandas.indexes.range': 'pandas.core.indexes.range',
               'pandas.indexes.numeric': 'pandas.core.indexes.base', 'pandas.indexes.multi': 'pandas.core.indexes.multi',
               'pandas.indexes': 'pandas.core.indexes', 'pandas.core.base': 'pandas.core.base',
               'pandas.core.index': 'pandas.core.indexes.base'}
    def find_class(self, module, name):
        module = self.RENAMES.get(module, module)
        if name in ('Int64Index', 'Float64Index', 'UInt64Index') and module.startswith('pandas'):
            return pd.Index
        return super().find_class(module, name)

def read_raw(path):
    errors = []
    try:
        return pd.read_pickle(path)
    except Exception as e:
        errors.append(f'read_pickle: {e!r}')
    try:
        import pandas.core.indexes as _idx
        sys.modules.setdefault('pandas.indexes', _idx)
        return pd.read_pickle(path)
    except Exception as e:
        errors.append(f'read_pickle with alias: {e!r}')
    try:
        with open(path, 'rb') as f:
            return _CompatUnpickler(f, encoding='latin1').load()
    except Exception as e:
        errors.append(f'compat unpickler: {e!r}')
    raise RuntimeError('Could not read LSWMD.pkl. Tried:\n  ' + '\n  '.join(errors) +
                       '\nIf all fail, create a separate environment with pandas==1.5.3, load the file there and re-save it with '
                       'df.to_pickle("data/LSWMD_new.pkl"), then set CONFIG["raw_file"] = "LSWMD_new.pkl".')

def kaggle_credentials():
    """Kaggle credentials, in order: environment variables, ~/.kaggle/kaggle.json, then a hidden prompt.
    Nothing is written to the notebook, so it is safe to share or push to GitHub."""
    user, key = os.environ.get('KAGGLE_USERNAME'), os.environ.get('KAGGLE_KEY') or os.environ.get('KAGGLE_API_TOKEN')
    if key: return user, key
    for p in (Path.home() / '.kaggle' / 'kaggle.json', HERE / 'kaggle.json'):
        if p.exists():
            j = json.loads(p.read_text()); return j.get('username'), j.get('key')
    import getpass
    print('Kaggle API key needed (kaggle.com -> Settings -> API -> Create New Token). Input is hidden and not saved.')
    user = input('Kaggle username: ').strip()
    key = getpass.getpass('Kaggle key: ').strip()
    return user, key

def download_from_kaggle(dataset=CONFIG['kaggle_dataset']):
    """Download the dataset zip through Kaggle's public API (no kaggle package needed) and extract the .pkl.
    The key is attached as an 'unredirected' header so it is not forwarded when Kaggle redirects to cloud storage."""
    import urllib.request, base64, zipfile, shutil
    user, key = kaggle_credentials()
    if not key: raise RuntimeError('No Kaggle key given.')
    req = urllib.request.Request(f'https://www.kaggle.com/api/v1/datasets/download/{dataset}')
    if key.startswith('KGAT_') or not user:      # newer single-string API tokens
        req.add_unredirected_header('Authorization', f'Bearer {key}')
    else:                                          # classic kaggle.json username + key
        req.add_unredirected_header('Authorization', 'Basic ' + base64.b64encode(f'{user}:{key}'.encode()).decode())
    zpath = DATA_DIR / 'wm811k.zip'
    print(f'Downloading {dataset} from Kaggle (large file; this can take several minutes)…')
    t = time.time()
    try:
        with urllib.request.urlopen(req) as r, open(zpath, 'wb') as f:
            shutil.copyfileobj(r, f, length=1 << 20)
    except urllib.error.HTTPError as e:
        zpath.unlink(missing_ok=True)
        hint = {401: 'wrong username or key', 403: 'open the dataset page once in a browser and accept its terms', 404: 'dataset not found'}.get(e.code, '')
        raise RuntimeError(f'Kaggle download failed: HTTP {e.code} {hint}') from None
    with zipfile.ZipFile(zpath) as zf:
        member = next(n for n in zf.namelist() if n.lower().endswith('.pkl'))
        with zf.open(member) as src, open(DATA_DIR / CONFIG['raw_file'], 'wb') as dst:
            shutil.copyfileobj(src, dst, length=1 << 20)
    zpath.unlink()
    print(f'Saved {CONFIG["raw_file"]} ({(DATA_DIR / CONFIG["raw_file"]).stat().st_size / 1e9:.2f} GB) in {time.time() - t:.0f} s')

def build_cache():
    t0 = time.time()
    raw_path = DATA_DIR / CONFIG['raw_file']
    if not raw_path.exists():
        raise FileNotFoundError(f'{raw_path} not found. Download LSWMD.pkl from Kaggle (see §2) and place it in {DATA_DIR}.')
    df = read_raw(raw_path)
    print(f'raw file loaded: {len(df):,} wafers in {time.time() - t0:.0f} s')
    meta = pd.DataFrame({
        'lot': df['lotName'].astype(str).to_numpy(),
        'slot': pd.to_numeric(df['waferIndex'], errors='coerce').to_numpy(),
        'die_size': pd.to_numeric(df['dieSize'], errors='coerce').to_numpy(),
        'label': [unpack(v) for v in df['failureType']],
        'orig_split': [unpack(v) for v in df['trianTestLabel']],
    })
    # keep every labelled map plus a fixed random sample of unlabeled maps; the rest are only summarised.
    # One pass over the maps, so no second full copy of the 811k arrays is ever held in memory.
    unl = np.where(meta['label'].to_numpy() == '')[0]
    keep = meta['label'].to_numpy() != ''
    if len(unl): keep[RNG.choice(unl, size=min(len(unl), 60_000), replace=False)] = True
    h, w, n_dies, n_fail, bad, hsh, kept = [], [], [], [], [], [], {}
    for i, m in enumerate(df['waferMap']):
        m = np.asarray(m)
        if m.ndim != 2: m = np.atleast_2d(m)
        h.append(m.shape[0]); w.append(m.shape[1])
        n_dies.append(int((m > 0).sum())); n_fail.append(int((m == 2).sum()))
        bad.append(int(((m != 0) & (m != 1) & (m != 2)).sum()))
        m8 = m.astype(np.uint8, copy=False)
        hsh.append(hashlib.md5(m8.tobytes() + str(m8.shape).encode()).hexdigest())
        if keep[i]: kept[i] = m8.copy()
        if (i + 1) % 200_000 == 0: print(f'  summarised {i + 1:,} maps…')
    meta['h'], meta['w'], meta['n_dies'], meta['n_fail'], meta['bad_values'], meta['map_hash'] = h, w, n_dies, n_fail, bad, hsh
    cache = {'meta': meta, 'maps': kept}
    del df; gc.collect()
    with open(DATA_DIR / CONFIG['cache_file'], 'wb') as f:
        pickle.dump(cache, f, protocol=4)
    print(f'cache written: {len(cache["maps"]):,} maps kept, {time.time() - t0:.0f} s in total')
    return cache

t0 = time.time()
cache_path = DATA_DIR / CONFIG['cache_file']
if not cache_path.exists() and not (DATA_DIR / CONFIG['raw_file']).exists():
    download_from_kaggle()
if cache_path.exists():
    with open(cache_path, 'rb') as f: cache = pickle.load(f)
    print(f'cache loaded in {time.time() - t0:.1f} s')
else:
    cache = build_cache()
meta, MAPS = cache['meta'], cache['maps']
meta['wafer_id'] = np.arange(len(meta))
print(f'{len(meta):,} wafers, {meta.lot.nunique():,} lots; {len(MAPS):,} maps held in memory')

## 4. Data-quality audit

The audit runs before any change. Each check asks whether a finding is a data problem or legitimate behaviour of the process.

### 4.1 Labels and the original split

In [ ]:
issues = []
meta['labelled'] = meta['label'] != ''
lab_counts = meta.loc[meta.labelled, 'label'].value_counts()
unknown = sorted(set(lab_counts.index) - set(CLASSES))
lab_tab = pd.DataFrame({'wafers': lab_counts, '% of labelled': 100 * lab_counts / lab_counts.sum()}).reindex(CLASSES)
lab_tab['% of defect wafers'] = 100 * lab_tab['wafers'] / lab_tab.loc[PATTERNS, 'wafers'].sum()
lab_tab.loc['none', '% of defect wafers'] = np.nan
display(lab_tab)
print(f'Labelled wafers: {meta.labelled.sum():,} of {len(meta):,} ({100 * meta.labelled.mean():.1f}%);  unlabeled: {(~meta.labelled).sum():,}')
print(f'Unexpected label values: {unknown if unknown else "none"}')
print('Original split among labelled wafers:', meta.loc[meta.labelled, 'orig_split'].value_counts().to_dict())
print(f'Imbalance: the largest defect class is {lab_tab.loc[PATTERNS, "wafers"].max() / lab_tab.loc[PATTERNS, "wafers"].min():.0f}x the smallest; '
      f'"none" is {100 * lab_tab.loc["none", "% of labelled"] / 100:.1f}% of labelled wafers')
issues.append(('Unlabelled majority', f'{100 * (~meta.labelled).mean():.0f}% of wafers have no label', 'Model on labelled; analyse unlabeled in §12',
               'Labelled wafers may not be representative'))
issues.append(('Class imbalance', f'"none" = {lab_tab.loc["none", "% of labelled"]:.0f}% of labels; smallest defect class {int(lab_tab.loc[PATTERNS, "wafers"].min())} wafers',
               'Macro-F1 and per-class recall; class weights; cap on "none"', 'Accuracy alone would be misleading'))

### 4.2 Map sizes, invalid values and tiny wafers

In [ ]:
bad_val_wafers = int((meta.bad_values > 0).sum())
tiny = meta.n_dies < CONFIG['min_dies']
size_ok = (meta.n_dies > 0) & (meta.h > 0) & (meta.w > 0)
fig, axes = plt.subplots(1, 3, figsize=(16, 3.8))
axes[0].hist(meta.n_dies, bins=80, color='#777777'); axes[0].axvline(CONFIG['min_dies'], color='#D55E00', ls='--', label=f'min {CONFIG["min_dies"]} dies')
axes[0].set(title='Dies per wafer', xlabel='dies', ylabel='wafers', xscale='log'); axes[0].legend()
shape_counts = meta.groupby(['h', 'w']).size().sort_values(ascending=False)
ks = min(15, len(shape_counts))
axes[1].bar(range(ks), shape_counts.head(ks).to_numpy(), color='#777777')
axes[1].set_xticks(range(ks)); axes[1].set_xticklabels([f'{h}x{w}' for h, w in shape_counts.head(ks).index], rotation=60, fontsize=7)
axes[1].set(title=f'Most common map sizes ({len(shape_counts):,} distinct)', ylabel='wafers')
ff = meta.n_fail / meta.n_dies.replace(0, np.nan)
for c in CLASSES:
    v = ff[meta.label == c].dropna()
    if len(v): axes[2].hist(v, bins=50, histtype='step', density=True, lw=1.4, color=PAL[c], label=c)
axes[2].set(title='Share of failing dies per wafer, by label', xlabel='failing-die share', ylabel='density'); axes[2].legend(fontsize=6, ncol=2)
fig.suptitle('Figure 4.1 — Map geometry and failure density', y=1.03); fig.tight_layout(); savefig(fig, '04_01_geometry')
print(f'Distinct map shapes: {len(shape_counts):,};  wafers with values other than 0/1/2: {bad_val_wafers};  '
      f'wafers with fewer than {CONFIG["min_dies"]} dies: {int(tiny.sum()):,} ({100 * tiny.mean():.2f}%)')
issues.append(('Variable map size', f'{len(shape_counts):,} distinct shapes', 'Features use normalised wafer coordinates', 'Models must be size-independent'))
issues.append(('Tiny wafers', f'{int(tiny.sum()):,} wafers with < {CONFIG["min_dies"]} dies', 'Exclude from modelling', 'Too coarse for spatial patterns'))

### 4.3 Duplicate maps and conflicting labels

Identical maps can legitimately occur (for example two clean wafers of the same product), but an identical map carrying **two different labels** is direct evidence of label noise: the same picture was judged differently.

In [ ]:
L = meta[meta.labelled]
dup_groups = L.groupby('map_hash')['label'].agg(['size', 'nunique'])
dup_maps = dup_groups[dup_groups['size'] > 1]
conflict = dup_groups[dup_groups['nunique'] > 1]
conf_rows = L[L.map_hash.isin(conflict.index)]
conf_pat = conf_rows[conf_rows.label != 'none']
print(f'Labelled maps that occur more than once: {len(dup_maps):,} distinct maps covering {int(dup_maps["size"].sum()):,} wafers')
print(f'Identical maps with conflicting labels: {len(conflict):,} groups ({len(conf_rows):,} wafers)')
if len(conflict):
    display(conf_rows.groupby('map_hash')['label'].apply(lambda s: ' / '.join(sorted(set(s)))).value_counts().head(8).rename('label combination'))
issues.append(('Duplicate maps', f'{int(dup_maps["size"].sum()):,} labelled wafers share a map with another wafer', 'Keep one per (map, lot); split by lot',
               'Duplicates across train/test would inflate accuracy'))
issues.append(('Conflicting labels', f'{len(conflict):,} identical maps carry different labels', 'Drop the conflicting wafers', 'An upper bound on achievable accuracy'))

### 4.4 Lots and slots

In [ ]:
lot_size = meta.groupby('lot').size()
lot_lab = meta.groupby('lot')['labelled'].mean()
slot_ok = meta.slot.between(1, 25)
print(f'Lots: {len(lot_size):,};  wafers per lot: median {lot_size.median():.0f}, max {lot_size.max()};  '
      f'lots with exactly 25 wafers: {100 * (lot_size == 25).mean():.1f}%')
print(f'Lots fully labelled: {(lot_lab == 1).sum():,};  partly labelled: {((lot_lab > 0) & (lot_lab < 1)).sum():,};  unlabeled: {(lot_lab == 0).sum():,}')
print(f'Slot index outside 1-25: {int((~slot_ok).sum())} wafers')
dup_slot = meta.duplicated(['lot', 'slot']).sum()
print(f'Duplicated (lot, slot) pairs: {int(dup_slot)}')
issues.append(('Lot structure', f'{(lot_lab == 1).sum():,} fully labelled lots', 'Lot-aware splits; lot analysis on fully labelled lots',
               'Wafers in a lot are not independent'))

In [ ]:
issue_table = pd.DataFrame(issues, columns=['issue', 'evidence', 'treatment', 'consequence'])
display(issue_table)

## 5. Cleaning decisions and cleaning log

No map is altered. Wafers are only *excluded* from the modelling set, for the reasons logged below. The lot and slot analyses in §8 use all labelled wafers, because they depend only on the labels.

In [ ]:
log = [('labelled wafers', int(meta.labelled.sum()))]
M = meta[meta.labelled & meta.label.isin(CLASSES)].copy(); log.append(('known label', len(M)))
M = M[(M.bad_values == 0) & size_ok.loc[M.index]]; log.append(('valid values and shape', len(M)))
M = M[M.n_dies >= CONFIG['min_dies']]; log.append((f'>= {CONFIG["min_dies"]} dies', len(M)))
M = M[~M.map_hash.isin(conflict.index)]; log.append(('drop conflicting-label duplicates', len(M)))
M = M.drop_duplicates(['map_hash', 'lot']); log.append(('one copy per (map, lot)', len(M)))
n_none_all = int((M.label == 'none').sum())
none_keep = M[M.label == 'none'].sample(n=min(CONFIG['max_none'], n_none_all), random_state=CONFIG['seed']).index
M = M[(M.label != 'none') | M.index.isin(none_keep)]; log.append((f'sample of "none" (cap {CONFIG["max_none"]:,})', len(M)))
clean_log = pd.DataFrame(log, columns=['step', 'wafers']); clean_log['removed'] = (-clean_log.wafers.diff()).fillna(0).astype(int)
display(clean_log)
display(M.label.value_counts().reindex(CLASSES).rename('wafers in modelling set').to_frame().T)
assert M.wafer_id.isin(list(MAPS.keys())).all(), 'every modelling wafer must have its map in memory'

## 6. What each pattern looks like

Six random examples per class from the modelling set (red = failing die, blue = passing die). Looking at real examples before modelling is essential: it shows how variable each class is and how blurry some boundaries are (a large `Loc` near the edge looks a lot like `Edge-Loc`).

In [ ]:
fig, axes = plt.subplots(len(CLASSES), 6, figsize=(10, 1.75 * len(CLASSES)))
for i, c in enumerate(CLASSES):
    ids = M.loc[M.label == c, 'wafer_id']
    ids = ids.sample(min(6, len(ids)), random_state=1).tolist() if len(ids) else []
    for j in range(6):
        ax = axes[i, j]
        if j < len(ids): show_wafer(ax, MAPS[ids[j]])
        else: ax.axis('off')
    axes[i, 0].set_ylabel(c, rotation=0, ha='right', va='center', fontsize=9, fontweight='bold')
    axes[i, 0].yaxis.set_label_coords(-0.1, 0.5)
fig.suptitle('Figure 6.1 — Example wafer maps per class', y=1.0); fig.tight_layout(); savefig(fig, '06_01_examples')

## 7. Q1 — Which patterns cost the most dies?

**Die yield** of a wafer = passing dies / all dies. Two views matter for prioritisation: how *severe* a pattern is on the wafers it hits (median yield), and how much *total* loss it causes across production, which also depends on how often it occurs. The total is measured as **excess failing dies**: failing dies above what the same wafer would have lost at the typical `none` failure rate, summed over all labelled wafers of that pattern.

In [ ]:
Lall = meta[meta.labelled & meta.label.isin(CLASSES) & (meta.n_dies > 0)].copy()
Lall['yield'] = 1 - Lall.n_fail / Lall.n_dies
base_fail = 1 - Lall.loc[Lall.label == 'none', 'yield'].median()
Lall['excess_fail'] = (Lall.n_fail - base_fail * Lall.n_dies).clip(lower=0)
yl = (Lall.groupby('label').agg(wafers=('yield', 'size'), median_yield=('yield', 'median'), mean_yield=('yield', 'mean'),
                                excess_fail_dies=('excess_fail', 'sum')).reindex(CLASSES))
yl['share of excess loss %'] = 100 * yl.excess_fail_dies / yl.loc[PATTERNS, 'excess_fail_dies'].sum()
yl.loc['none', 'share of excess loss %'] = np.nan
pareto = yl.loc[PATTERNS].sort_values('excess_fail_dies', ascending=False)
pareto['cumulative %'] = pareto['share of excess loss %'].cumsum()
display(pareto.round(3))

fig, axes = plt.subplots(1, 2, figsize=(15, 4.4))
order_y = yl.sort_values('median_yield').index.tolist()
sns.boxplot(data=Lall, x='label', y='yield', order=order_y, palette=PAL, ax=axes[0], fliersize=1, linewidth=0.8)
axes[0].set(title='Die yield per wafer by pattern', xlabel='', ylabel='die yield'); axes[0].tick_params(axis='x', rotation=30)
ax = axes[1]
ax.bar(pareto.index, pareto['share of excess loss %'], color=[PAL[c] for c in pareto.index], edgecolor='k', lw=0.5)
ax2 = ax.twinx(); ax2.plot(pareto.index, pareto['cumulative %'], 'k-o', ms=4); ax2.set_ylim(0, 105); ax2.grid(False); ax2.set_ylabel('cumulative %')
ax.set(title='Pareto of excess die loss across labelled production', ylabel='% of excess failing dies'); ax.tick_params(axis='x', rotation=30)
fig.suptitle('Figure 7.1 — Severity vs total impact of each pattern', y=1.03); fig.tight_layout(); savefig(fig, '07_01_yield_pareto')
top2 = pareto.index[:2].tolist()
note(f'**Reading Figure 7.1.** Severity and total impact rank the patterns differently. **{order_y[0]}** is the most severe '
     f'(median yield {yl.loc[order_y[0], "median_yield"]:.0%}), but the largest *total* loss comes from **{top2[0]}** '
     f'({pareto.loc[top2[0], "share of excess loss %"]:.0f}% of excess failing dies) and **{top2[1]}** '
     f'({pareto.loc[top2[1], "share of excess loss %"]:.0f}%), together {pareto.loc[top2[1], "cumulative %"]:.0f}%. '
     f'A yield-improvement programme that started with the most dramatic-looking pattern would be working on the wrong problem. '
     f'The `none` wafers have a median yield of {yl.loc["none", "median_yield"]:.0%}, which is the baseline used for "excess" loss.')

## 8. Q2 — Are defects systematic? Lot clustering and slot position

**Lot clustering.** If a pattern is caused by a tool or recipe problem, it tends to hit several wafers of the same lot, because those wafers were processed together. If it is random, a wafer's lot-mates are no more likely to share its pattern than any other wafer. For each pattern, the **lot clustering lift** compares the share of a pattern wafer's lot-mates that have the same pattern with the pattern's overall rate. Lift ≈ 1 means random; lift ≫ 1 means systematic. The test uses only fully labelled lots, and its p-value comes from shuffling labels across lots 500 times.

In [ ]:
full_lots = lot_lab[lot_lab == 1].index
FL = meta[meta.lot.isin(full_lots) & meta.label.isin(CLASSES)][['lot', 'slot', 'label']].copy()
lot_codes = FL.lot.astype('category').cat.codes.to_numpy()

def same_share(labels, codes, cls):
    is_c = (labels == cls).astype(int)
    per_lot = np.bincount(codes, weights=is_c); n_lot = np.bincount(codes)
    k = per_lot[codes]; n = n_lot[codes]
    m = (is_c == 1) & (n > 1)
    return ((k[m] - 1) / (n[m] - 1)).mean() if m.any() else np.nan

lab_arr = FL.label.to_numpy(); rows = []
for c in PATTERNS:
    base = (lab_arr == c).mean()
    if base == 0: continue
    obs = same_share(lab_arr, lot_codes, c)
    null = np.array([same_share(RNG.permutation(lab_arr), lot_codes, c) for _ in range(500)])
    rows.append({'pattern': c, 'wafers': int((lab_arr == c).sum()), 'overall rate %': 100 * base,
                 'lot-mates with same pattern %': 100 * obs, 'lift': obs / base, 'p (permutation)': (1 + (null >= obs).sum()) / 501})
lotc = pd.DataFrame(rows).sort_values('lift', ascending=False); lotc['q'] = bh(lotc['p (permutation)'])
display(lotc.round(3))
print(f'Fully labelled lots used: {len(full_lots):,} ({len(FL):,} wafers)')

**Slot position.** Wafers sit in fixed slots (1–25) in the carrier and are often processed in slot order. A pattern that is more frequent at the first or last slots hints at a chamber warm-up or seasoning effect, or a handling problem at one end of the cassette. For each pattern, a χ² test compares its rate across five slot groups (1–5, 6–10, …, 21–25); p-values are corrected with Benjamini-Hochberg.

In [ ]:
S = meta[meta.labelled & meta.label.isin(CLASSES) & meta.slot.between(1, 25)].copy()
S['slot_group'] = pd.cut(S.slot, [0, 5, 10, 15, 20, 25], labels=['1-5', '6-10', '11-15', '16-20', '21-25'])
rows = []
for c in PATTERNS:
    tab = pd.crosstab(S.slot_group, S.label == c)
    if tab.shape[1] < 2: continue
    chi2, p, dof, _ = stats.chi2_contingency(tab)
    rate = 100 * tab[True] / tab.sum(axis=1)
    rows.append({'pattern': c, 'chi2': chi2, 'p': p, **{f'rate % slots {g}': rate[g] for g in rate.index},
                 'max/min rate': rate.max() / rate.min() if rate.min() > 0 else np.nan})
slot_tab = pd.DataFrame(rows); slot_tab['q'] = bh(slot_tab.p)
display(slot_tab.sort_values('p').round(3))

fig, axes = plt.subplots(1, 2, figsize=(15, 4))
ax = axes[0]
ax.barh(lotc.pattern, lotc.lift, color=[PAL[c] for c in lotc.pattern], edgecolor='k', lw=0.5)
ax.axvline(1, color='k', ls='--'); ax.set(title='Lot clustering lift (1 = random)', xlabel='lift'); ax.invert_yaxis()
ax = axes[1]
srate = S.groupby('slot')['label'].apply(lambda s: (s != 'none').mean() * 100)
ax.plot(srate.index, srate.to_numpy(), 'o-', color='#0072B2')
ax.set(title='Share of labelled wafers with any defect pattern, by slot', xlabel='slot (waferIndex)', ylabel='% defective'); ax.set_xticks(range(1, 26, 2))
fig.suptitle('Figure 8.1 — Systematic vs random behaviour', y=1.03); fig.tight_layout(); savefig(fig, '08_01_lot_slot')
sys_pat = lotc[(lotc.q < CONFIG['fdr_alpha']) & (lotc.lift > 2)].pattern.tolist()
slot_pat = slot_tab[slot_tab.q < CONFIG['fdr_alpha']].pattern.tolist()
note(f'**Reading Figure 8.1.** Patterns that cluster strongly within lots (lift > 2, q < 0.05): **{", ".join(sys_pat) if sys_pat else "none"}**. '
     f'These behave like *systematic* problems: when one wafer shows them, its lot-mates are much more likely to as well, so the lot\'s tool '
     f'history is the first place to look. Patterns with a significant slot dependence (q < 0.05): **{", ".join(slot_pat) if slot_pat else "none"}**. '
     f'A slot effect suggests something that changes along the cassette (first-wafer effects, chamber seasoning, a handling step), and is '
     f'worth checking against the order in which slots were processed.')

## 9. Feature engineering: turning a map into numbers an engineer can read

Each map is converted to a fixed set of size-independent features. Every feature has a physical reading, so a prediction can be explained in the language of the fab ("failures concentrated beyond 85% of the radius"), which matters when the result has to convince a customer.

| Family | Features | Captures |
|---|---|---|
| Overall | failing-die share | Severity (`Near-full`, `Random`) |
| Radial profile | fail rate in 10 rings from centre to edge | `Center`, `Donut`, `Edge-Ring` |
| 13-region density | fail rate in a 3×3 inner grid + 4 edge sectors (Wu et al. 2015) | Where on the wafer |
| Angular concentration | max and spread of fail rate over 8 angular sectors of the outer zone | Full ring (`Edge-Ring`) vs one sector (`Edge-Loc`) |
| Radon projections | mean and spread over 60 angles of the Radon transform, resampled to 20 points | Lines (`Scratch`), symmetric shapes |
| Largest failing cluster | area, perimeter, major/minor axis, eccentricity, solidity, distance from centre (after a 3×3 median filter) | Shape and position of the main defect |
| Spatial clustering | share of failing dies whose neighbours also fail, relative to random | `Random` vs clustered patterns |

In [ ]:
NR, NP = CONFIG['radon_angles'], CONFIG['radon_points']
THETA = np.linspace(0., 180., NR, endpoint=False)
KERNEL = np.ones((3, 3)); KERNEL[1, 1] = 0

def features(m):
    on = m > 0; fail = (m == 2).astype(float); n_on = on.sum()
    yy, xx = np.nonzero(on); cy, cx = yy.mean(), xx.mean(); R = np.sqrt(n_on / np.pi)
    Y, Xg = np.mgrid[:m.shape[0], :m.shape[1]]
    r = np.hypot(Y - cy, Xg - cx) / R; th = np.arctan2(Y - cy, Xg - cx)
    f = {'fail_share': fail[on].mean()}
    rb = np.minimum((r * 10).astype(int), 9)
    for k in range(10):
        sel = on & (rb == k); f[f'ring_{k}'] = fail[sel].mean() if sel.any() else 0.0
    u, v = (Xg - cx) / R, (Y - cy) / R
    inner = (np.abs(u) < 0.6) & (np.abs(v) < 0.6)
    gi = np.clip(((u + 0.6) / 0.4).astype(int), 0, 2); gj = np.clip(((v + 0.6) / 0.4).astype(int), 0, 2)
    for i in range(3):
        for j in range(3):
            sel = on & inner & (gi == i) & (gj == j); f[f'grid_{j}{i}'] = fail[sel].mean() if sel.any() else 0.0
    quad = ((th + np.pi / 4) % (2 * np.pi) // (np.pi / 2)).astype(int)
    for q in range(4):
        sel = on & ~inner & (quad == q); f[f'edge_{q}'] = fail[sel].mean() if sel.any() else 0.0
    outer = on & (r > 0.7); sec = ((th + np.pi) / (2 * np.pi) * 8).astype(int) % 8
    srates = np.array([fail[outer & (sec == s)].mean() if (outer & (sec == s)).any() else 0.0 for s in range(8)])
    f['outer_sector_max'] = srates.max(); f['outer_sector_cv'] = srates.std() / (srates.mean() + 1e-6)
    f['outer_sector_min'] = srates.min()
    sino = radon(fail, theta=THETA, circle=False)
    for name, prof in (('radon_mean', sino.mean(axis=1)), ('radon_std', sino.std(axis=1))):
        xs = np.linspace(0, 1, len(prof)); vals = interp1d(xs, prof, kind='cubic')(np.linspace(0, 1, NP))
        vals = vals / (R + 1e-6)
        for k in range(NP): f[f'{name}_{k}'] = vals[k]
    den = median_filter((m == 2).astype(np.uint8), size=3) & on
    lab_img = cc_label(den, connectivity=2)
    props = max(regionprops(lab_img), key=lambda p: p.area) if lab_img.max() > 0 else None
    if props is not None:
        f.update({'cl_area': props.area / n_on, 'cl_perimeter': props.perimeter / R, 'cl_major': props.major_axis_length / R,
                  'cl_minor': props.minor_axis_length / R, 'cl_ecc': props.eccentricity, 'cl_solidity': props.solidity,
                  'cl_dist': np.hypot(props.centroid[0] - cy, props.centroid[1] - cx) / R, 'n_clusters': lab_img.max()})
    else:
        f.update({'cl_area': 0, 'cl_perimeter': 0, 'cl_major': 0, 'cl_minor': 0, 'cl_ecc': 0, 'cl_solidity': 0, 'cl_dist': 0, 'n_clusters': 0})
    nb_fail = convolve(fail, KERNEL, mode='constant'); nb_on = convolve(on.astype(float), KERNEL, mode='constant')
    fm = (fail == 1) & on
    obs = (nb_fail[fm] / np.maximum(nb_on[fm], 1)).mean() if fm.any() else 0.0
    f['neighbour_fail'] = obs; f['clustering_ratio'] = obs / (f['fail_share'] + 1e-6)
    return f

def features_batch(maps):
    return [features(m) for m in maps]

def extract(ids):
    # each worker receives only its own batch of maps (never the whole MAPS dictionary)
    chunks = np.array_split(np.asarray(ids), max(1, len(ids) // 500))
    try:
        res = Parallel(n_jobs=CONFIG['n_jobs'])(delayed(features_batch)([MAPS[i] for i in c]) for c in chunks)
    except Exception as e:
        print(f'parallel extraction failed ({e!r}); running sequentially'); res = [features_batch([MAPS[i] for i in c]) for c in chunks]
    return pd.DataFrame([r for part in res for r in part], index=np.asarray(ids))

t0 = time.time()
F = extract(M.wafer_id.to_numpy())
FEATS = list(F.columns)
print(f'{F.shape[1]} features for {len(F):,} wafers in {time.time() - t0:.0f} s;  any missing: {bool(F.isna().any().any())}')
F.round(5).assign(label=M.set_index('wafer_id').label).to_csv(OUT / 'features.csv.gz')

**Do the features carry the physics we expect?** Class-average radial profiles and a few headline features:

In [ ]:
Fl = F.join(M.set_index('wafer_id')[['label', 'lot']])
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
ring_cols = [f'ring_{k}' for k in range(10)]
for c in CLASSES:
    axes[0].plot(np.arange(10) / 10 + 0.05, Fl.loc[Fl.label == c, ring_cols].mean().to_numpy(), '-o', ms=3, color=PAL[c], label=c, lw=1.5 if c != 'none' else 2.5)
axes[0].set(title='Mean radial fail-rate profile', xlabel='normalised radius (0 = centre, 1 = edge)', ylabel='fail rate'); axes[0].legend(fontsize=7, ncol=2)
sns.boxplot(data=Fl, x='label', y='outer_sector_cv', order=CLASSES, palette=PAL, ax=axes[1], fliersize=0, linewidth=0.8)
axes[1].set(title='Angular concentration at the edge', xlabel='', ylabel='CV of outer-sector fail rates'); axes[1].tick_params(axis='x', rotation=45)
axes[1].set_ylim(0, Fl.outer_sector_cv.quantile(0.98))
sns.boxplot(data=Fl, x='label', y='cl_ecc', order=CLASSES, palette=PAL, ax=axes[2], fliersize=0, linewidth=0.8)
axes[2].set(title='Elongation of the largest failing cluster', xlabel='', ylabel='eccentricity'); axes[2].tick_params(axis='x', rotation=45)
fig.suptitle('Figure 9.1 — Feature sanity check: each family separates the classes it was designed for', y=1.03); fig.tight_layout(); savefig(fig, '09_01_feature_check')
kw = {c: stats.kruskal(*[Fl.loc[Fl.label == k, c] for k in CLASSES if (Fl.label == k).sum() > 5]).statistic for c in FEATS}
top_kw = pd.Series(kw).sort_values(ascending=False)
cm_ = Fl.groupby('label')[ring_cols + ['outer_sector_cv', 'cl_ecc']].median()
mid = Fl.groupby('label')[[f'ring_{k}' for k in (3, 4, 5)]].mean().mean(axis=1)
checks = [('highest centre fail rate (ring 0)', 'Center', Fl.groupby('label')['ring_0'].mean().drop(['Near-full', 'Random'], errors='ignore').idxmax()),
          ('highest edge fail rate (ring 9)', 'Edge-Ring', Fl.groupby('label')['ring_9'].mean().drop(['Near-full', 'Random'], errors='ignore').idxmax()),
          ('highest mid-radius fail rate (rings 3-5)', 'Donut', mid.drop(['Near-full', 'Random'], errors='ignore').idxmax()),
          ('most concentrated edge failures (sector CV)', 'Edge-Loc', cm_['outer_sector_cv'].idxmax()),
          ('most elongated largest cluster', 'Scratch', cm_['cl_ecc'].idxmax())]
rows_ = [f'| {what} | `{exp}` | `{got}` | {"yes" if exp == got else "**no**"} |' for what, exp, got in checks]
note('**Reading Figure 9.1 — do the features behave as designed?**\n\n| Check | Expected class | Observed class | As expected? |\n|---|---|---|---|\n' +
     '\n'.join(rows_) + f'\n\n{sum(e == g for _, e, g in checks)} of {len(checks)} design checks hold. '
     f'The most discriminating single features (Kruskal-Wallis H) are {", ".join(f"`{c}`" for c in top_kw.index[:5])}. '
     'A check that fails is worth looking at in the example maps of §6 before trusting the model.')

## 10. Q3 — Can a model recognise the patterns on unseen lots?

### 10.1 Lot-aware split
Wafers of one lot share history and often look alike (§8). If lot-mates were split between training and test, the model could recognise *the lot* rather than the pattern. All splits here are made over **lots**: 20% of lots form a sealed test set, 15% a validation set for choosing between models, and the rest train.

In [ ]:
groups = Fl.lot.to_numpy(); yall = Fl.label.to_numpy()
gss = GroupShuffleSplit(n_splits=1, test_size=CONFIG['test_lots'], random_state=CONFIG['seed'])
dev_i, test_i = next(gss.split(Fl, yall, groups))
gss2 = GroupShuffleSplit(n_splits=1, test_size=CONFIG['val_lots'] / (1 - CONFIG['test_lots']), random_state=CONFIG['seed'] + 1)
tr_j, va_j = next(gss2.split(Fl.iloc[dev_i], yall[dev_i], groups[dev_i]))
tr_i, va_i = dev_i[tr_j], dev_i[va_j]
assert not (set(groups[tr_i]) & set(groups[test_i])) and not (set(groups[va_i]) & set(groups[test_i])) and not (set(groups[tr_i]) & set(groups[va_i]))
split_tab = pd.DataFrame({s: pd.Series(yall[i]).value_counts().reindex(CLASSES).fillna(0).astype(int)
                          for s, i in [('train', tr_i), ('validation', va_i), ('test', test_i)]}).T
split_tab['lots'] = [len(set(groups[i])) for i in (tr_i, va_i, test_i)]
display(split_tab)
Xf = Fl[FEATS].to_numpy()

def sample_weights(yv):
    cnt = Counter(yv); return np.array([len(yv) / (len(cnt) * cnt[v]) for v in yv])

def evaluate(yt, yp):
    return {'macro-F1': f1_score(yt, yp, average='macro', labels=CLASSES, zero_division=0),
            'balanced acc': balanced_accuracy_score(yt, yp), 'accuracy': accuracy_score(yt, yp),
            'macro-F1 (defects only)': f1_score(yt, yp, average='macro', labels=PATTERNS, zero_division=0)}

### 10.2 Candidate models, compared on the validation lots

* **Majority class**: always predicts `none`. The floor.
* **Pixel baseline**: multinomial logistic regression on the failing-die map resized to 32×32. Shows what plain pixels achieve without spatial features.
* **Logistic regression on features**: a linear model on the engineered features.
* **Random Forest** and **Gradient Boosting** on the features, with balanced class weights (Gradient Boosting stops early once an internal 10% hold-out stops improving).

In [ ]:
def pixels(ids, n=CONFIG['pixel_size']):
    out = np.empty((len(ids), n * n), np.float32)
    for k, i in enumerate(ids):
        m = MAPS[i]; out[k] = resize((m == 2).astype(float), (n, n), order=1, anti_aliasing=True, preserve_range=True).ravel()
    return out

t0 = time.time()
P_all = pixels(Fl.index.to_numpy())
models = {
    'Pixel baseline (logistic, 32x32)': ('pix', make_pipeline(StandardScaler(), LogisticRegression(C=0.05, max_iter=300, class_weight='balanced'))),
    'Logistic regression (features)': ('feat', make_pipeline(StandardScaler(), LogisticRegression(C=1.0, max_iter=2000, class_weight='balanced'))),
    'Random Forest (features)': ('feat', RandomForestClassifier(n_estimators=300, min_samples_leaf=2, class_weight='balanced_subsample',
                                                                n_jobs=CONFIG['n_jobs'], random_state=CONFIG['seed'])),
    'Gradient Boosting (features)': ('feat', HistGradientBoostingClassifier(max_iter=200, learning_rate=0.1, max_leaf_nodes=31,
                                                                            l2_regularization=1.0, early_stopping=True, n_iter_no_change=10,
                                                                            validation_fraction=0.1, random_state=CONFIG['seed'])),
}
val_rows, fitted = [{'model': 'Majority class (none)', **evaluate(yall[va_i], np.full(len(va_i), 'none'))}], {}
for name, (kind, mdl) in models.items():
    A = P_all if kind == 'pix' else Xf; t1 = time.time()
    if 'Gradient' in name: mdl.fit(A[tr_i], yall[tr_i], sample_weight=sample_weights(yall[tr_i]))
    else: mdl.fit(A[tr_i], yall[tr_i])
    fitted[name] = (kind, mdl)
    val_rows.append({'model': name, **evaluate(yall[va_i], mdl.predict(A[va_i])), 'fit s': time.time() - t1})
val_tab = pd.DataFrame(val_rows)
display(val_tab.round(3))
best_name = val_tab.iloc[1:].sort_values('macro-F1', ascending=False).model.iloc[0]
print(f'Selected on validation macro-F1: {best_name}   ({time.time() - t0:.0f} s for all candidates)')

### 10.3 How much would a random split have flattered the model?
The same Random Forest is trained once with a random wafer-level split and once with the lot-aware split, using the same number of training wafers. The difference is the optimism you would report by ignoring lots.

In [ ]:
rf_proto = models['Random Forest (features)'][1]
ridx = np.arange(len(Fl))
r_tr, r_te = train_test_split(ridx, test_size=len(test_i), stratify=yall, random_state=CONFIG['seed'])
r_tr = r_tr[:len(tr_i)]
rf_rand = sklearn.base.clone(rf_proto).fit(Xf[r_tr], yall[r_tr])
leak_tab = pd.DataFrame([
    {'split': 'random (wafer-level)', **evaluate(yall[r_te], rf_rand.predict(Xf[r_te])),
     'test wafers with a lot-mate in training %': 100 * np.isin(groups[r_te], groups[r_tr]).mean()},
    {'split': 'lot-aware', **evaluate(yall[test_i], fitted['Random Forest (features)'][1].predict(Xf[test_i])),
     'test wafers with a lot-mate in training %': 100 * np.isin(groups[test_i], groups[tr_i]).mean()}])
display(leak_tab.round(3))

### 10.4 One evaluation on the sealed test lots
The selected model is retrained on train + validation lots and evaluated once on the test lots. Per-class precision and recall matter more than the average: a model that misses most `Scratch` wafers is useless for catching handling damage, whatever its overall accuracy.

In [ ]:
kind, proto = fitted[best_name]
A = P_all if kind == 'pix' else Xf
final = sklearn.base.clone(proto)
if 'Gradient' in best_name: final.fit(A[dev_i], yall[dev_i], sample_weight=sample_weights(yall[dev_i]))
else: final.fit(A[dev_i], yall[dev_i])
pred = final.predict(A[test_i]); proba = final.predict_proba(A[test_i])
test_metrics = evaluate(yall[test_i], pred)
display(pd.DataFrame([{'model': best_name, **test_metrics}]).round(3))
rep = pd.DataFrame(classification_report(yall[test_i], pred, labels=CLASSES, output_dict=True, zero_division=0)).T.loc[CLASSES]
rep['support'] = rep['support'].astype(int)
display(rep.round(3))
rep.to_csv(OUT / 'test_per_class_report.csv')

cm = confusion_matrix(yall[test_i], pred, labels=CLASSES)
cmn = cm / cm.sum(axis=1, keepdims=True).clip(min=1)
fig, ax = plt.subplots(figsize=(8.5, 7))
sns.heatmap(cmn, annot=cm, fmt='d', cmap='Blues', xticklabels=CLASSES, yticklabels=CLASSES, cbar_kws={'label': 'share of true class'}, ax=ax, annot_kws={'fontsize': 8})
ax.set(xlabel='predicted', ylabel='true', title=f'{best_name} on sealed test lots (colour = row share, numbers = wafers)')
fig.suptitle('Figure 10.1 — Confusion matrix', y=1.0); fig.tight_layout(); savefig(fig, '10_01_confusion')
off = pd.DataFrame(cm, index=CLASSES, columns=CLASSES); np.fill_diagonal(off.values, 0) if False else None
pairs = [(a, b, cm[i, j], cmn[i, j]) for i, a in enumerate(CLASSES) for j, b in enumerate(CLASSES) if i != j and cm[i, j] > 0]
pairs = sorted(pairs, key=lambda t: -t[2])[:5]
worst = rep.loc[PATTERNS, 'recall'].idxmin(); bestc = rep.loc[PATTERNS, 'recall'].idxmax()
note(f'**Reading Figure 10.1.** On lots the model never saw, macro-F1 is **{test_metrics["macro-F1"]:.3f}** and balanced accuracy '
     f'**{test_metrics["balanced acc"]:.3f}**. The best-recognised pattern is **{bestc}** (recall {rep.loc[bestc, "recall"]:.0%}); the weakest '
     f'is **{worst}** (recall {rep.loc[worst, "recall"]:.0%}). The most frequent confusions are: ' +
     ('; '.join(f'{a} → {b} ({n} wafers, {s:.0%} of {a})' for a, b, n, s in pairs) if pairs else 'none') +
     '. Compare these pairs with the pattern descriptions in §1: confusions between neighbouring patterns (for example `Loc` and `Edge-Loc`) '
     'are expected, while confusions between unrelated patterns usually point to label noise (§11.1).')

## 11. Error analysis and what the model relies on

### 11.1 Confident mistakes
The wafers below are test wafers the model got wrong *with high confidence*. Looking at them is the fastest way to tell model weaknesses from label noise: many confident "errors" in WM-811K are wafers whose expert label is debatable.

In [ ]:
conf = proba.max(axis=1); wrong = np.where(pred != yall[test_i])[0]
top_wrong = wrong[np.argsort(-conf[wrong])][:12]
fig, axes = plt.subplots(2, 6, figsize=(13, 5))
for ax, k in zip(axes.ravel(), top_wrong):
    show_wafer(ax, MAPS[Fl.index[test_i[k]]], f'true {yall[test_i][k]}\npred {pred[k]} ({conf[k]:.0%})')
for ax in axes.ravel()[len(top_wrong):]: ax.axis('off')
fig.suptitle('Figure 11.1 — The 12 most confident errors on the test lots', y=1.0); fig.tight_layout(); savefig(fig, '11_01_errors')

### 11.2 Which feature families matter?
Permutation importance on the test lots, grouped by feature family: all features of a family are shuffled together, and the drop in macro-F1 is recorded. Grouping avoids the usual trap where correlated features (for example neighbouring rings) share credit and each looks unimportant.

In [ ]:
FAMILIES = {'overall fail share': ['fail_share'], 'radial profile': [c for c in FEATS if c.startswith('ring_')],
            '13-region density': [c for c in FEATS if c.startswith(('grid_', 'edge_'))],
            'angular concentration': [c for c in FEATS if c.startswith('outer_sector')],
            'Radon projections': [c for c in FEATS if c.startswith('radon_')],
            'largest cluster geometry': [c for c in FEATS if c.startswith('cl_') or c == 'n_clusters'],
            'spatial clustering': ['neighbour_fail', 'clustering_ratio']}
if kind == 'feat':
    Xt = Xf[test_i]; base_f1 = f1_score(yall[test_i], final.predict(Xt), average='macro', labels=CLASSES, zero_division=0)
    rows = []
    for fam, cols in FAMILIES.items():
        idx = [FEATS.index(c) for c in cols]; drops = []
        for rep_ in range(5):
            Xp = Xt.copy(); perm = RNG.permutation(len(Xp)); Xp[:, idx] = Xp[perm][:, idx]
            drops.append(base_f1 - f1_score(yall[test_i], final.predict(Xp), average='macro', labels=CLASSES, zero_division=0))
        rows.append({'family': fam, 'features': len(cols), 'macro-F1 drop': np.mean(drops), 'sd': np.std(drops)})
    fam_imp = pd.DataFrame(rows).sort_values('macro-F1 drop', ascending=False)
    display(fam_imp.round(4))
    # per-class view: which family matters most for each pattern's recall
    per = {}
    for fam, cols in FAMILIES.items():
        idx = [FEATS.index(c) for c in cols]; Xp = Xt.copy(); Xp[:, idx] = Xp[RNG.permutation(len(Xp))][:, idx]
        pp = final.predict(Xp)
        per[fam] = {c: (pred[yall[test_i] == c] == c).mean() - (pp[yall[test_i] == c] == c).mean() for c in CLASSES if (yall[test_i] == c).any()}
    per = pd.DataFrame(per)
    fig, ax = plt.subplots(figsize=(10, 5))
    sns.heatmap(per, annot=True, fmt='.2f', cmap='Reds', ax=ax, cbar_kws={'label': 'drop in recall when family is shuffled'})
    ax.set(title='Which feature family each class depends on', xlabel='', ylabel=''); ax.tick_params(axis='x', rotation=30)
    fig.suptitle('Figure 11.2 — Feature-family importance by class', y=1.0); fig.tight_layout(); savefig(fig, '11_02_family_importance')
    lead = per.idxmax(axis=1); lead_v = per.max(axis=1)
    parts = [f'`{c}` → *{lead[c]}* (recall drop {lead_v[c]:.2f})' if lead_v[c] >= 0.05 else f'`{c}` → no single family dominates'
             for c in PATTERNS if c in lead.index]
    note('**Reading Figure 11.2.** The family each pattern depends on most: ' + '; '.join(parts) +
         '. Compare this with the physics in §1 and §9 (for example, `Edge-Ring` should depend on the radial profile and `Scratch` on cluster '
         'geometry or Radon features). Where the model\'s reasoning matches the physics, its predictions are easier to defend to a customer; '
         'where it does not, the model may be exploiting a shortcut in the labels.')
else:
    print('Selected model uses pixels; feature-family importance not applicable.')

## 12. Q4 — What does the model see in unlabeled production wafers?

About four in five wafers in WM-811K were never labelled. Applying the final model to a random sample of them estimates how common each pattern is in production. Two cautions: the model was trained with a capped number of `none` wafers, so its raw probabilities over-predict defects; and only predictions above a confidence threshold are counted. The estimate is therefore a screening tool for which unlabeled wafers to review, not an exact census.

In [ ]:
U = meta[(~meta.labelled) & meta.wafer_id.isin(list(MAPS.keys())) & (meta.n_dies >= CONFIG['min_dies']) & (meta.bad_values == 0)]
U = U.sample(n=min(CONFIG['n_unlabeled'], len(U)), random_state=CONFIG['seed'])
t0 = time.time()
FU = extract(U.wafer_id.to_numpy())
AU = FU[FEATS].to_numpy() if kind == 'feat' else pixels(U.wafer_id.to_numpy())
pu = final.predict_proba(AU); cls_order = list(final.classes_)
# prior correction: rescale probabilities from the training class mix to the true labelled class mix (Saerens et al. 2002)
train_prior = pd.Series(yall[dev_i]).value_counts(normalize=True).reindex(cls_order).to_numpy()
true_prior = lab_counts.reindex(cls_order).fillna(0).to_numpy(); true_prior = true_prior / true_prior.sum()
adj = pu * (true_prior / train_prior); adj = adj / adj.sum(axis=1, keepdims=True)
top = adj.argmax(axis=1); topp = adj.max(axis=1)
pred_u = np.array(cls_order)[top]
confident = topp >= CONFIG['confidence']
est = pd.DataFrame({'predicted (all)': pd.Series(pred_u).value_counts(), 'predicted (confident)': pd.Series(pred_u[confident]).value_counts()}).reindex(CLASSES).fillna(0).astype(int)
est['% of confident'] = 100 * est['predicted (confident)'] / est['predicted (confident)'].sum()
est['% of labelled set'] = 100 * lab_counts.reindex(CLASSES) / lab_counts.sum()
print(f'{len(U):,} unlabeled wafers scored in {time.time() - t0:.0f} s; {100 * confident.mean():.1f}% above the {CONFIG["confidence"]:.0%} confidence threshold')
display(est.round(2))

fig, axes = plt.subplots(2, 8, figsize=(15, 4.4))
for j, c in enumerate(PATTERNS):
    cand = np.where((pred_u == c) & confident)[0]
    cand = cand[np.argsort(-topp[cand])][:2]
    for i in range(2):
        ax = axes[i, j]
        if i < len(cand): show_wafer(ax, MAPS[U.wafer_id.iloc[cand[i]]], f'{c} ({topp[cand[i]]:.0%})' if i == 0 else f'{topp[cand[i]]:.0%}')
        else: ax.axis('off')
fig.suptitle('Figure 12.1 — Most confident pattern predictions on unlabeled production wafers', y=1.02); fig.tight_layout(); savefig(fig, '12_01_unlabeled')
pd.DataFrame({'wafer_id': U.wafer_id.to_numpy(), 'lot': U.lot.to_numpy(), 'pred': pred_u, 'confidence': topp}).to_csv(OUT / 'unlabeled_predictions.csv', index=False)

## 13. Answers, recommendations and limitations

In [ ]:
q1 = f'**{pareto.index[0]}** and **{pareto.index[1]}** cause {pareto["cumulative %"].iloc[1]:.0f}% of excess die loss; **{order_y[0]}** is the most severe per wafer.'
q2 = (f'Lot clustering (lift > 2, q < 0.05): {", ".join(sys_pat) if sys_pat else "none"}. Slot dependence (q < 0.05): {", ".join(slot_pat) if slot_pat else "none"}.')
q3 = (f'Yes, with caveats. {best_name} reaches macro-F1 {test_metrics["macro-F1"]:.3f} on unseen lots (pixel baseline on validation: '
      f'{val_tab.set_index("model").loc["Pixel baseline (logistic, 32x32)", "macro-F1"]:.3f}). Weakest class: {worst} (recall {rep.loc[worst, "recall"]:.0%}). '
      f'Same-size comparison of splits (§10.3): random {leak_tab["macro-F1"].iloc[0]:.3f} vs lot-aware {leak_tab["macro-F1"].iloc[1]:.3f}, '
      f'so ignoring lots overstates macro-F1 by {leak_tab["macro-F1"].iloc[0] - leak_tab["macro-F1"].iloc[1]:+.3f}.')
q4 = f'{100 * confident.mean():.0f}% of sampled unlabeled wafers get a confident prediction; the most common predicted defect is **{est.loc[PATTERNS, "predicted (confident)"].idxmax()}**.'
note('| Question | Answer |\n|---|---|\n' + f'| Q1 Which patterns cost the most dies? | {q1} |\n| Q2 Systematic or random? | {q2} |\n'
     f'| Q3 Recognisable on unseen lots? | {q3} |\n| Q4 What is in unlabeled production? | {q4} |')

**Recommendations for a fab engineering team**
1. **Prioritise by total loss, not by how dramatic a pattern looks.** Use the Pareto in §7 to pick the first two patterns to attack.
2. **Treat lot-clustered patterns as tool/process problems.** For the patterns flagged in §8, pull the tool and chamber history of affected lots and compare it with clean lots; that is where a root cause is most likely to be found.
3. **Deploy the classifier as a triage tool.** Route confident predictions straight to the responsible module owner, and send low-confidence wafers and the weakest classes to a human reviewer.
4. **Fix the labels before buying a bigger model.** Identical maps with different labels (§4.3) and the confident errors in §11.1 show that label noise limits accuracy. A short relabelling pass on the confused pairs would likely gain more than a more complex model.

**Limitations**
* No process or tool history is available, so root causes are *hypotheses* to investigate, not conclusions.
* The model only knows the 8 labelled patterns; mixed patterns (two defects on one wafer) and new patterns will be forced into one class.
* `none` was sub-sampled for speed, so probabilities were prior-corrected in §12; production prevalence estimates are approximate.
* Lot clustering is measured on fully labelled lots only, which may not be representative of all production.

**Natural next steps:** a small CNN on the 2-channel map (needs PyTorch, ideally a GPU) to compare against the feature model; similarity search ("find past wafers that look like this one"), as proposed by Wu et al.; and a mixed-pattern dataset such as MixedWM38.

## 14. Reproducibility record

In [ ]:
headline = {
    'wafers_total': len(meta), 'lots_total': int(meta.lot.nunique()), 'labelled': int(meta.labelled.sum()),
    'modelling_set': len(Fl), 'features': len(FEATS),
    'conflicting_label_groups': len(conflict),
    'pareto_top2': pareto.index[:2].tolist(), 'pareto_top2_share_pct': round(float(pareto['cumulative %'].iloc[1]), 1),
    'lot_clustered_patterns': sys_pat, 'slot_dependent_patterns': slot_pat,
    'selected_model': best_name, **{f'test_{k}': round(v, 4) for k, v in test_metrics.items()},
    'random_split_macro_f1': round(float(leak_tab['macro-F1'].iloc[0]), 4),
    'unlabeled_scored': len(U), 'unlabeled_confident_pct': round(100 * confident.mean(), 1),
    'versions': {'python': platform.python_version(), 'numpy': np.__version__, 'pandas': pd.__version__, 'sklearn': sklearn.__version__, 'skimage': skimage.__version__},
}
json.dump(headline, open(OUT / 'headline_numbers.json', 'w'), indent=1, default=str)
for k, v in headline.items(): print(f'{k:<32} {v}')